In [1]:
import pandas as pd
from rdkit import Chem

# --- 1. Define Classification Logic ---
def classify_component(smiles):
    """
    Classifies a single molecular component.
    """
    if pd.isna(smiles) or smiles == "":
        return "Invalid", "Empty"
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid", "RDKit parse error"

    # Pattern: Alpha-Beta Unsaturated Ester/Acid (C=C-C(=O)O)
    acrylate_pattern = Chem.MolFromSmarts('[CX3]=[CX3][CX3](=[OX1])[OX2]')
    
    if not mol.HasSubstructMatch(acrylate_pattern):
        return "Not Acrylate", "No acrylic moiety"

    matches = mol.GetSubstructMatches(acrylate_pattern)
    
    has_acrylate = False
    has_methacrylate = False
    has_beta_sub = False # Cinnamate-like
    has_alpha_sub = False
    
    for match in matches:
        beta_idx, alpha_idx, carbonyl_idx = match[0], match[1], match[2]
        beta_atom = mol.GetAtomWithIdx(beta_idx)
        alpha_atom = mol.GetAtomWithIdx(alpha_idx)
        
        # Check if terminal alkene (Beta has 2 Hydrogens)
        if beta_atom.GetTotalNumHs() == 2:
            # Check Alpha substitution
            if alpha_atom.GetTotalNumHs() == 1:
                has_acrylate = True # Standard Acrylate
            else:
                # Check if the substituent is Methyl (Methacrylate)
                is_methyl = False
                for neighbor in alpha_atom.GetNeighbors():
                    nid = neighbor.GetIdx()
                    if nid not in [beta_idx, carbonyl_idx]:
                        if neighbor.GetAtomicNum() == 6 and neighbor.GetTotalNumHs() == 3:
                            is_methyl = True
                
                if is_methyl:
                    has_methacrylate = True
                else:
                    has_alpha_sub = True
        else:
            has_beta_sub = True # Internal alkene (Cinnamate, etc)

    # Priority Classification
    if has_acrylate: return "Acrylate", "Terminal CH2=CH-COO"
    if has_methacrylate: return "Methacrylate", "Terminal CH2=C(Me)-COO"
    if has_alpha_sub: return "Alpha-Substituted", "Terminal CH2=C(R)-COO"
    if has_beta_sub: return "Beta-Substituted", "Internal double bond"
    
    return "Uncertain", "Pattern matched but logic unclear"

# --- 2. Load Data ---
df = pd.read_csv('acrylates.csv')

# --- 3. Separate Mixtures (The "Explode" Step) ---
# Split the SMILES string by '.' into a list of strings
df['smiles_component'] = df['smiles'].str.split('.')

# Explode the lists into separate rows
# (The original 'id' and 'cmpdname' are duplicated for each component)
df_separated = df.explode('smiles_component')

# --- 4. Classify Each Component ---
print("Classifying components...")
results = df_separated['smiles_component'].apply(classify_component)

df_separated['Category'] = [r[0] for r in results]
df_separated['Reason'] = [r[1] for r in results]

# --- 5. Save Result ---
df_separated = df_separated.drop(columns=['smiles']).rename(columns={'smiles_component':'smiles'}).drop_duplicates(subset='smiles')
output_filename = 'acrylates_classified.csv'
df_separated.to_csv(output_filename, index=False)

Classifying components...


[21:21:00] WARNING: not removing hydrogen atom without neighbors


In [3]:
df_separated[df_separated['Category'] == 'Acrylate'].drop(columns=['Category', 'Reason']).to_csv('../polygraphpy/data/full_dataset.csv', index=False)